In [0]:
#Get paths and client key variables from Databricks job variables after file-reader notebook runs
dbutils.widgets.text("wc_loss_run_path", "")
dbutils.widgets.text("ttd_path", "")
dbutils.widgets.text("ptr_path", "")
dbutils.widgets.text("liab_loss_run_path", "")
dbutils.widgets.text("client_key", "")

wc_loss_run_path   = dbutils.widgets.get("wc_loss_run_path")
ttd_path           = dbutils.widgets.get("ttd_path")
ptr_path           = dbutils.widgets.get("ptr_path")
liab_loss_run_path = dbutils.widgets.get("liab_loss_run_path")
client_key         = dbutils.widgets.get("client_key")

In [0]:
#Define paths for various directories. Run mkdirs command to ensure directories exist
INPROGRESS_DIR = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/in_progress/"
PROCESSED_DIR  = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/processed"
FAILED_DIR     = "/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/failed"

for p in [INPROGRESS_DIR, PROCESSED_DIR, FAILED_DIR]:
   dbutils.fs.mkdirs(p)

In [0]:
#Define file-mover function
def move(src, dst_dir):
   dst = f"{dst_dir.rstrip('/')}/{src.split('/')[-1]}"
   dbutils.fs.mv(src, dst)
   return dst

In [0]:
#Moves files from input folder to in_progress folder as they are collected for each run. Prevents files from being processed multiple times
claimed = {}

if wc_loss_run_path:
   claimed["wc_loss_run_path"] = move(wc_loss_run_path, INPROGRESS_DIR)
if ttd_path:
   claimed["ttd_path"] = move(ttd_path, INPROGRESS_DIR)
if ptr_path:
   claimed["ptr_path"] = move(ptr_path, INPROGRESS_DIR)
if liab_loss_run_path:
   claimed["liab_loss_run_path"] = move(liab_loss_run_path, INPROGRESS_DIR)

In [0]:
#Runs Transformation notebook using claimed files as inputs. If Transformation fails, files are moved to failed folder. If Transformation succeeds, files are moved to processed folder.
try:
   dbutils.notebook.run(
       "/Workspace/Users/jonah.crook@aon.com/Triage/Triage_Transformation_Output_Consultant_DQ_Loop",
       timeout_seconds=0,
       arguments={
           "wc_loss_run_path": claimed.get("wc_loss_run_path", ""),
           "ttd_path": claimed.get("ttd_path", ""),
           "ptr_path": claimed.get("ptr_path", ""),
           "liab_loss_run_path": claimed.get("liab_loss_run_path", ""),
           "client_key": client_key,
       }
   )
   for p in claimed.values():
       move(p, PROCESSED_DIR)
except Exception:
   for p in claimed.values():
       move(p, FAILED_DIR)
   raise